In [0]:
# Importing Libraries
from delta.tables import DeltaTable
from pyspark.sql import  SparkSession
spark = SparkSession.builder \
    .appName("Production_ETL") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()


# Func to create,insert or upsert
def upsert_to_gold(df, target_schema, target_table, join_key):
    full_table_path = f"{target_schema}.{target_table}"
    if not spark.catalog.tableExists(full_table_path):
        print(f"🚀 Table {full_table_path} does not exist. Creating new Delta table...")
        df.write.format("delta") \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(full_table_path)
    else:
        print(f"🔄 Table {full_table_path} exists. Performing Delta Merge (Upsert)...")
        target_delta_table = DeltaTable.forName(spark, full_table_path)
        (target_delta_table.alias("target")
            .merge(
                df.alias("source"),
                f"target.{join_key} = source.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        print(f"✅ Upsert successful for {target_table}")


In [0]:
silver_df=spark.read.table("instagram.silverlayer.silver_instagram_users")

In [0]:
# %sql
# drop table instagram.goldlayer.fact_user_engagement;

In [0]:
# Columns in my tables
fact_engagement_df = silver_df.select(
    "user_id", 
    "daily_active_minutes_instagram", 
    "time_on_reels_per_day", 
    "time_on_feed_per_day", 
    "time_on_messages_per_day", 
    "time_on_explore_per_day", 
    "user_engagement_score", 
    "sessions_per_day","posts_created_per_week","reels_watched_per_day","stories_viewed_per_day","likes_given_per_day","comments_written_per_day","dms_sent_per_week","dms_received_per_week","ads_clicked_per_day","ads_viewed_per_day"
).distinct()

upsert_to_gold(
    df=fact_engagement_df, 
    target_schema="instagram.goldlayer", 
    target_table="fact_user_engagement", 
    join_key="user_id"
)